<a href="https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/hw/HW/2026/01-agentic-rag/hw-01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Homework 1]('https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/hw/HW/2026/01-agentic-rag/hw-01.ipynb')

In [ ]:
%pip install requests minsearch groq python-dotenv gitsource

## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [ ]:
print(f"Q1: {len(documents)}")

Q1: 72


## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [ ]:
import minsearch
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"

In [ ]:
index.search(query, num_results=1)



[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [ ]:
print(f"Q2: 01-agentic-rag/lessons/14-agentic-loop.md")

Q2: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

# Завантаження змінних середовища з файлу .env
load_dotenv()

# Убедитесь, что переменная окружения GROQ_API_KEY установлена
api_key_groq = os.environ.get("GROQ_API_KEY")

if api_key_groq is None:
    raise ValueError("API key is not set. Please set the GROQ_API_KEY environment variable.")

client_groq = Groq(api_key=api_key_groq)

In [ ]:
chat_completion = client_groq.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.1-8b-instant",
)

print(chat_completion.choices[0].message.content)

Fast language models, also known as efficient or lightweight language models, have gained significant attention in recent years due to their importance in various applications. Here are some reasons why fast language models are crucial:

1. **Real-time Processing**: Fast language models can process inputs in real-time, allowing for immediate responses, which is essential for applications like chatbots, voice assistants, and language translation services.
2. **Scalability**: Efficient language models can handle large volumes of data and scale well with increasing input sizes, making them suitable for applications like search engine ranking and language modeling for large datasets.
3. **Power Efficiency**: Fast language models require less computational power compared to traditional language models, making them more suitable for battery-powered devices like smartphones and smart speakers.
4. **Reduced Memory Requirements**: Fast language models typically have smaller memory requirements,

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [ ]:
from dataclasses import dataclass

@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int

class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        # ⚠️ Тут потрібно адаптувати під вашу схему (filename/content)
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response


    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        # Витягуємо текст і usage правильно
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        return RAGResult(answer=answer, input_tokens=input_tokens, output_tokens=output_tokens)


In [ ]:
rag_helper = RAGBase(index=index, llm_client=client_groq, model="llama-3.3-70b-versatile")
result = rag_helper.rag(query)
print(resultQ.answer)
Q3_input_tokens = result.input_tokens
print("Input tokens:", Q3_input_tokens)


Please go ahead and ask your question based on the provided context. I'll answer according to the information given in the text. If the answer is not found in the context, I will respond with "I don't know."
Input tokens: 7221


In [ ]:
print(f"""Q3:
input tokens: {Q3_input_tokens}
output tokens: {result.output_tokens}""")

Q3: 
input tokens: 7221
output tokens: 313


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

In [ ]:
print(f"Q4: {len(chunks)}")

Q4: 295


## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

# алгоритм дій:
1. Розбиваємо документи на chunks.
2. Індексуємо chunks замість цілих документів.
3. Пошук повертає лише ті chunks, де є релевантний текст.
4. Формуємо prompt із цих chunks.

In [ ]:
import minsearch
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

In [ ]:
rag_helper = RAGBase(index=chunk_index, llm_client=client_groq, model="llama-3.3-70b-versatile")
result = rag_helper.rag(query)
print(result.answer)
Q5_input_tokens = result.input_tokens
print("Input tokens:", Q5_input_tokens)
print("Output tokens:", result.output_tokens)

The agentic loop keeps calling the model until it stops by using a `while True` loop that continues to iterate until the model returns a response without any function calls. This is achieved by checking the `has_function_calls` flag, which is set to `True` if the model output contains any function calls. If `has_function_calls` is `False`, the loop breaks, indicating that the model has returned a final answer with no more tool calls.

In the provided code, the loop iterates as follows:

1. It prints the current iteration number.
2. It sends a request to the model with the current messages and tools.
3. It processes the model's response, checking for function calls and messages.
4. If a function call is found, it runs the corresponding tool and appends the output to the messages.
5. It increments the iteration counter and checks the `has_function_calls` flag.
6. If `has_function_calls` is `False`, the loop breaks, and the final answer is returned.

This process allows the agentic loop t

In [ ]:
print(f"Q5: {int(Q3_input_tokens/Q5_input_tokens)}x fewer")

Q5: 3x fewer


## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

In [ ]:
%pip install toyaikit

In [ ]:
import json

search_call_count = 0

def search(query: str):
    """Search the course lessons for relevant content."""
    global search_call_count
    search_call_count += 1

    results = chunk_index.search(query, num_results=5)
    return results

def make_call(call):
    """Execute a function call and return the output."""
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lessons for relevant content",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

search_call_count = 0
it = 1
last_answer = ""

total_input_tokens = 0
total_output_tokens = 0

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=messages,
        tools=[search_tool]
    )

    # Accumulate token usage
    total_input_tokens += response.usage.input_tokens
    total_output_tokens += response.usage.output_tokens

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print(f"function_call: {item.name} {item.arguments}")
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            last_answer = item.content[0].text
            print(last_answer)

    it = it + 1
    if has_function_calls == False:
        break

print(f"\n--- Number of Search Calls ---")
print(f"The agent called search {search_call_count} times")

# Example pricing for Groq's llama-3.3-70b-versatile (check Groq's official pricing for actual values)
# Assuming: Input token price = $0.70 per 1M tokens, Output token price = $0.80 per 1M tokens
price_per_million_input_tokens = 0.70
price_per_million_output_tokens = 0.80

total_cost = (
    (total_input_tokens / 1_000_000) * price_per_million_input_tokens +
    (total_output_tokens / 1_000_000) * price_per_million_output_tokens
)

print(f"\n--- Token Usage and Cost ---")
print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")
print(f"Estimated Total Cost: ${total_cost:.4f} (based on example pricing)")


iteration #1...
function_call: search {"query":"agentic loop vs plain RAG"}
iteration #2...
ASSISTANT:
The agentic loop works by having the model decide which tools to call, and then sending the tool results back to the model to produce a proper course-specific answer. This process involves more round-trips compared to plain RAG, where only one call is made. The agentic loop is also known as "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same, where the LLM decides which tools to call. The cost of using the agentic loop is higher than plain RAG, as it involves multiple API calls, each of which costs money. The cost can be calculated by converting token counts to dollars using the price per million input tokens and per million output tokens.

--- Number of Search Calls ---
The agent called search 1 times


# ToyAIKit

Видео: [Смотреть этот урок](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Написанный вручную агентный цикл из предыдущего урока полезен для обучения, но он довольно однообразен. Каждый раз при создании нового агента вам пришлось бы писать один и тот же цикл `while`, одну и ту же обработку вызовов функций и управление сообщениями.

`ToyAIKit` инкапсулирует этот паттерн, позволяя вам сосредоточиться на инструментах, промптах и поведении. Мы построили его вместе на одном из воркшопов DataTalks.Club некоторое время назад. Он делает то же самое, что и наш ручной цикл, но с меньшим количеством шаблонного кода. Если вы заглянете в код `runners`, вы найдете там тот же цикл `while True`, который мы писали сами.

Я использую его здесь намеренно, потому что не хочу выбирать победителя среди промышленных фреймворков. `ToyAIKit` маленький и легко читается, поэтому, если что-то сломается, вы сможете увидеть, что именно произошло. Это делает его удобным для разработки и отладки локально перед переходом в продакшн.

Одно предостережение: `ToyAIKit` — это библиотека для обучения и экспериментов, она НЕ предназначена для использования в реальных проектах (production). Мы используем ее, потому что она минималистична и наглядна.

## Настройка

Установите библиотеку:

```bash
uv add toyaikit
```

Импортируйте необходимые классы:

```python
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
```

## Регистрация инструмента

Мы регистрируем нашу функцию `search` вместе со схемой из предыдущих уроков:

```python
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
```

## Автоматическая генерация схемы в ToyAIKit

Писать схему вручную утомительно, и мы не хотим делать это для каждой функции. На самом деле, это и не обязательно.

Если мы добавим аннотации типов и строку документации (docstring) к функции `search`, `ToyAIKit` прочитает их и сам составит схему за нас:

```python
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )
```

Затем зарегистрируйте ее без передачи схемы:

```python
agent_tools = Tools()
agent_tools.add_tool(search)
```

Вы можете посмотреть, что сгенерировал `ToyAIKit`:

```python
agent_tools.get_tools()
```

На выходе вы получите ту же JSON-схему, которую мы писали вручную в уроке про вызов функций. `ToyAIKit` создал ее на основе докстринга и аннотации типа.

Любой современный агентный фреймворк использует этот прием. Он считывает типизированную функцию Python с документацией и строит на ее основе схему. Так работают OpenAI Agents SDK, PydanticAI, LangChain и Google ADK. Вы пишете инструмент, а фреймворк сам понимает, как его описать.

## Интерфейс чата и раннер (runner)

Создайте интерфейс чата и колбэк, а затем постройте раннер:

```python
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
```

`chat_interface` отвечает за отображение в ноутбуке. `callback` отрисовывает сообщения модели и вызовы инструментов по мере их возникновения. Раннер запускает агентный цикл — тот самый `while True`, который мы писали вручную. Он отправляет сообщения, выполняет вызовы функций, добавляет результаты инструментов обратно и повторяет процесс до тех пор, пока модель не закончит.

Мы намеренно выбрали здесь `gpt-5.4-mini`. Без нее `ToyAIKit` использует более простую и быструю модель по умолчанию, которая не так надежно следует инструкциям.

## Запуск одного промпта

Запустите один запрос:

```python
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)
```

Мы специально использовали опечатку «Olama». Агент выполняет поиск, получает плохие результаты, а затем пробует еще раз с «Ollama». Механизм восстановления такой же, как в ручном цикле. Вывод в ноутбуке выглядит гораздо приятнее: каждый вызов инструмента и сообщение отображаются по очереди, так что вы можете изучить каждый результат поиска.

`result` — это объект `LoopResult`, содержащий `all_messages` (всю историю диалога), количество токенов и `cost` (стоимость, рассчитанную на основе использования токенов).

## Стоимость и токены

Посмотрите, сколько стоил этот вызов:

```python
result.cost
```

Это полезно при разработке — особенно для многоходовых агентов, где один запрос может вызвать несколько обращений к модели. В ручном цикле вам приходилось считать это самим. Фреймворк делает это за вас автоматически.

Вы также можете просмотреть полную историю сообщений:

```python
result.all_messages
```

Это просто список — тот же список `messages`, который мы вели вручную.

## Продолжение диалога

Возьмите сообщения из предыдущего результата и передайте их как `previous_messages` при следующем вызове `loop`:

```python
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)
```

Раннер продолжит с того места, где остановился предыдущий вызов, используя тот же агентный цикл и расширенную историю. Модель поймет, что под «другой моделью» подразумевается Ollama, так как она видит предыдущий ход в памяти. Без этой истории она бы понятия не имела, о чем идет речь.

## Интерактивный чат

Для работы в режиме чата запустите встроенный цикл ввода:

```python
runner.run()
```

Вводите вопросы и получайте ответы. Введите «stop», чтобы выйти.

[← Агентный цикл](14-agentic-loop.md) | [Другие фреймворки →](16-other-frameworks.md)
